Copyright 2026 Snowflake Inc.
SPDX-License-Identifier: Apache-2.0

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.

# Module 2: Taking Advantage of Iceberg Tables

This notebook contains code examples for Module 2 videos.

⚠️ **IMPORTANT**: Before running cells in this notebook, open and run `Videos/Setup.ipynb` to download the required data.

This environment uses **Spark Connect** — a single Spark server runs in the background and all notebooks connect to it as lightweight clients. If you see `ConnectionRefusedError: [Errno 111] Connection refused` when initializing the Spark Session, the Spark Connect server may not be running. Check the Docker container logs with `docker logs jupyter-spark`. If the server failed to start, try restarting the container with `docker compose restart jupyter`.

## Setup

Initialize Spark session for Module 2 examples.

In [2]:
# Initialize Spark session for Module 2 examples

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("Module2_Taking_Advantage") \
    .getOrCreate()


/opt/conda/lib/python3.13/site-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at spark/connect/base.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
/opt/conda/lib/python3.13/site-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at spark/connect/commands.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
/opt/conda/lib/python3.13/site-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at spark/connect/common.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
/opt/conda/lib/p

## Video: Moving existing data to Iceberg

In [3]:
# Configure RustFS client for S3 operations

import boto3
from botocore.client import Config

s3_client = boto3.client(
    's3',
    endpoint_url='http://rustfs:9000',
    aws_access_key_id='admin',
    aws_secret_access_key='password',
    config=Config(signature_version='s3v4'),
    region_name='us-east-1'
)

### Snapshot Procedure

Use snapshot to create Iceberg metadata over existing Parquet files without rewriting data.


In [4]:
# Clean up any November data to ensure it's not present during snapshot

# Delete November file from trips_snapshot/data if it exists
try:
    s3_client.delete_object(Bucket='warehouse', Key='taxi/trips_snapshot/data/yellow_tripdata_2023-11.parquet')
    print("Deleted November data from trips_snapshot/data/")
except:
    print("November data not present in trips_snapshot/data/")

Deleted November data from trips_snapshot/data/


### View Raw Data Files

[Open trips_snapshot in RustFS](http://localhost:9001/browser/warehouse/taxi/trips_snapshot/)

Login: `admin` / `password`

The table's metadata is in `/metadata/` and data files are in `/data/` - both under the same table location.

In [5]:
# Create temporary table in session catalog, then snapshot it to create Iceberg metadata

# Drop existing snapshot table if it exists
spark.sql("DROP TABLE IF EXISTS polaris.taxi.trips_snapshot")

# Switch to session catalog and create a table that references the Parquet files
spark.sql("USE spark_catalog")

spark.sql("""
    CREATE TABLE IF NOT EXISTS default.temp_parquet_source
    USING parquet
    LOCATION 's3a://warehouse/taxi/trips_snapshot/data/'
""")

# Snapshot with location at taxi/trips_snapshot - data files are already in place
spark.sql("""
    CALL polaris.system.snapshot(
        source_table => 'default.temp_parquet_source',
        table => 'polaris.taxi.trips_snapshot',
        location => 's3a://warehouse/taxi/trips_snapshot'
    )
""").show()

# Switch back to polaris catalog
spark.sql("USE polaris")

# Clean up the temporary table
spark.sql("DROP TABLE IF EXISTS spark_catalog.default.temp_parquet_source")

+--------------------+
|imported_files_count|
+--------------------+
|                   5|
+--------------------+



DataFrame[]

### View Snapshot Table in RustFS

[Open trips_snapshot in RustFS](http://localhost:9001/browser/warehouse/taxi/trips_snapshot/)

Login: `admin` / `password`

The table's metadata is in `/metadata/` and data files are in `/data/` - both under the same table location.

In [6]:
# Verify the snapshot created successfully and view file metadata

spark.sql("""
    SELECT 
        COUNT(*) as total_trips,
        MIN(tpep_pickup_datetime) as first_pickup,
        MAX(tpep_pickup_datetime) as last_pickup
    FROM polaris.taxi.trips_snapshot
""").show()

spark.sql("""
    SELECT 
        file_path,
        ROUND(file_size_in_bytes / 1024 / 1024, 2) as size_mb,
        record_count
    FROM polaris.taxi.trips_snapshot.files
""").show(truncate=False)

+-----------+-------------------+-------------------+
|total_trips|       first_pickup|        last_pickup|
+-----------+-------------------+-------------------+
|   15407558|2002-12-31 22:27:05|2023-11-23 13:40:43|
+-----------+-------------------+-------------------+

+------------------------------------------------------------------------+-------+------------+
|file_path                                                               |size_mb|record_count|
+------------------------------------------------------------------------+-------+------------+
|s3a://warehouse/taxi/trips_snapshot/data/yellow_tripdata_2023-06.parquet|52.45  |3307234     |
|s3a://warehouse/taxi/trips_snapshot/data/yellow_tripdata_2023-07.parquet|46.12  |2907108     |
|s3a://warehouse/taxi/trips_snapshot/data/yellow_tripdata_2023-08.parquet|45.92  |2824209     |
|s3a://warehouse/taxi/trips_snapshot/data/yellow_tripdata_2023-09.parquet|45.68  |2846722     |
|s3a://warehouse/taxi/trips_snapshot/data/yellow_tripdata

### Download an additional month of taxi data

In [7]:
# Download November 2023 taxi data for add_files demonstration

import urllib.request
import os

# Download November data directly into the table's directory
url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-11.parquet"
filename = "yellow_tripdata_2023-11.parquet"
local_path = f"/tmp/{filename}"
s3_key = f"taxi/trips_snapshot/data/{filename}"

print(f"Downloading {filename}...")
urllib.request.urlretrieve(url, local_path)

print(f"Uploading to RustFS: {s3_key}")
s3_client.upload_file(local_path, 'warehouse', s3_key)

os.remove(local_path)
print(f"Uploaded {filename} to s3://warehouse/taxi/trips_snapshot/data/")

Uploading to RustFS: taxi/trips_snapshot/data/yellow_tripdata_2023-11.parquet
Uploaded yellow_tripdata_2023-11.parquet to s3://warehouse/taxi/trips_snapshot/data/


In [8]:
# Use add_files to incrementally add new Parquet files to an existing Iceberg table

spark.sql("""
    CALL polaris.system.add_files(
        table => 'polaris.taxi.trips_snapshot',
        source_table => 'parquet.`s3a://warehouse/taxi/trips_snapshot/data/yellow_tripdata_2023-11.parquet`'
    )
""").show()

+-----------------+-----------------------+
|added_files_count|changed_partition_count|
+-----------------+-----------------------+
|                1|                   NULL|
+-----------------+-----------------------+



In [9]:
# Check the files metadata table to see the new files

spark.sql("""
    SELECT COUNT(*) FROM polaris.taxi.trips_snapshot.files
""").show()


+--------+
|count(1)|
+--------+
|       6|
+--------+



In [10]:
# Verify the incremental file additions worked correctly

spark.sql("""
    SELECT 
        COUNT(*) as total_trips,
        MIN(tpep_pickup_datetime) as first_pickup,
        MAX(tpep_pickup_datetime) as last_pickup
    FROM polaris.taxi.trips_snapshot
""").show()

+-----------+-------------------+-------------------+
|total_trips|       first_pickup|        last_pickup|
+-----------+-------------------+-------------------+
|   18747273|2002-12-31 22:27:05|2023-12-01 00:03:36|
+-----------+-------------------+-------------------+



### Reserialization with CTAS

When data isn't in a compatible format (CSV, JSON, etc.) or you need to transform during migration, use CREATE TABLE AS SELECT. This is compute-intensive but provides complete flexibility.

In [11]:
import tempfile
import os

csv_data = """VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,fare_amount,total_amount
1,2023-06-01 10:15:00,2023-06-01 10:30:00,2,3.5,15.0,20.5
2,2023-06-01 11:20:00,2023-06-01 11:45:00,1,5.2,22.0,28.0
1,2023-06-01 14:30:00,2023-06-01 15:00:00,3,7.8,30.0,38.5
"""

temp_dir = "/tmp/csv_demo"
os.makedirs(temp_dir, exist_ok=True)
csv_path = os.path.join(temp_dir, "sample_trips.csv")

with open(csv_path, 'w') as f:
    f.write(csv_data)

In [12]:
# Migrate CSV data to Iceberg using CTAS with proper schema transformation

# First, read the CSV with headers
csv_df = spark.read.format("csv") \
    .option("header", "true") \
    .load("file:///tmp/csv_demo/sample_trips.csv")

# Create temporary view for CTAS
csv_df.createOrReplaceTempView("csv_source")

# Now create the Iceberg table from the CSV
spark.sql("""
    CREATE TABLE IF NOT EXISTS polaris.taxi.trips_from_csv
    USING iceberg
    PARTITIONED BY (days(tpep_pickup_datetime))
    TBLPROPERTIES (
        'format-version' = '3'
    )
    AS 
    SELECT 
        CAST(VendorID AS INT) as VendorID,
        CAST(tpep_pickup_datetime AS TIMESTAMP) as tpep_pickup_datetime,
        CAST(tpep_dropoff_datetime AS TIMESTAMP) as tpep_dropoff_datetime,
        CAST(passenger_count AS INT) as passenger_count,
        CAST(trip_distance AS DOUBLE) as trip_distance,
        CAST(fare_amount AS DOUBLE) as fare_amount,
        CAST(total_amount AS DOUBLE) as total_amount
    FROM csv_source
""")

DataFrame[]

In [13]:
# View our CSV Data now in an Iceberg Table

spark.sql("SELECT * FROM polaris.taxi.trips_from_csv").show()

+--------+--------------------+---------------------+---------------+-------------+-----------+------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|fare_amount|total_amount|
+--------+--------------------+---------------------+---------------+-------------+-----------+------------+
|       1| 2023-06-01 10:15:00|  2023-06-01 10:30:00|              2|          3.5|       15.0|        20.5|
|       2| 2023-06-01 11:20:00|  2023-06-01 11:45:00|              1|          5.2|       22.0|        28.0|
|       1| 2023-06-01 14:30:00|  2023-06-01 15:00:00|              3|          7.8|       30.0|        38.5|
+--------+--------------------+---------------------+---------------+-------------+-----------+------------+



## Video: Git-like features with Write-Audit-Publish and Branching and Tagging


In [14]:
# Create trips_by_month table with WAP enabled for Write-Audit-Publish pattern
spark.sql(""" DROP TABLE IF EXISTS polaris.taxi.trips_by_month """)
spark.sql("""
    CREATE TABLE IF NOT EXISTS polaris.taxi.trips_by_month
    USING iceberg
    PARTITIONED BY (months(tpep_pickup_datetime))
    TBLPROPERTIES (
        'format-version' = '3',
        'write.wap.enabled' = 'true'
    )
    AS
    SELECT * FROM parquet.`s3a://warehouse/raw/`
""")
spark.sql("SELECT COUNT(*) from polaris.taxi.trips_by_month").show()

+--------+
|count(1)|
+--------+
|15407558|
+--------+



In [15]:
# Add a new column to the schema
# In this example, we'll add a correct_fare column (half of the incorrect total_amount)

spark.sql("""
    ALTER TABLE polaris.taxi.trips_by_month
    ADD COLUMN correct_fare DOUBLE
""")

DataFrame[]

In [16]:
# Enable Write-Audit-Publish (WAP) and perform UPDATE operation
# Set a wap.id to identify this staged write operation
# The changes will be written but not immediately visible to normal queries

spark.conf.set("spark.wap.id", "wap-demo-001")

spark.sql("""
    UPDATE polaris.taxi.trips_by_month
    SET correct_fare = total_amount / 2.0
""")

# Unset the wap.id to ensure subsequent operations are not part of this WAP operation
spark.conf.unset("spark.wap.id")

In [17]:
# Query the production table WITHOUT the WAP snapshot - correct_fare should be NULL
print("\nData in production table (WITHOUT WAP changes - correct_fare is NULL):")
spark.sql("""
    SELECT tpep_pickup_datetime, total_amount, correct_fare
    FROM polaris.taxi.trips_by_month
    LIMIT 10
""").show()


Data in production table (WITHOUT WAP changes - correct_fare is NULL):
+--------------------+------------+------------+
|tpep_pickup_datetime|total_amount|correct_fare|
+--------------------+------------+------------+
| 2023-11-01 19:37:07|       46.85|        NULL|
| 2023-11-02 01:52:29|        13.6|        NULL|
| 2023-11-02 04:06:10|       101.5|        NULL|
| 2023-11-23 10:34:56|       72.35|        NULL|
| 2023-11-02 06:40:10|        22.4|        NULL|
| 2023-11-23 13:40:43|       36.12|        NULL|
| 2023-11-01 00:00:27|        61.0|        NULL|
| 2023-11-01 00:00:04|       21.36|        NULL|
| 2023-11-01 00:04:51|        34.8|        NULL|
| 2023-11-01 00:14:15|       21.36|        NULL|
+--------------------+------------+------------+



In [19]:
# Find the snapshot_id that corresponds to our wap.id
# This snapshot contains the staged changes

from pyspark.sql.functions import col

wap_snapshot_id = spark.sql("""
    SELECT snapshot_id 
    FROM polaris.taxi.trips_by_month.snapshots
    WHERE summary['wap.id'] = 'wap-demo-001'
""").collect()[0]['snapshot_id']

print(f"WAP Snapshot ID: {wap_snapshot_id}")

# Query the WAP snapshot to audit the changes before publishing
# Use VERSION AS OF syntax with snapshot ID to see the staged data
print("\nData in WAP snapshot (with changes):")
spark.sql(f"""
    SELECT tpep_pickup_datetime, total_amount, correct_fare
    FROM polaris.taxi.trips_by_month VERSION AS OF {wap_snapshot_id}
    LIMIT 10
""").show()


WAP Snapshot ID: 3487655135651362189

Data in WAP snapshot (with changes):
+--------------------+------------+------------+
|tpep_pickup_datetime|total_amount|correct_fare|
+--------------------+------------+------------+
| 2023-11-01 19:37:07|       46.85|      23.425|
| 2023-11-02 01:52:29|        13.6|         6.8|
| 2023-11-02 04:06:10|       101.5|       50.75|
| 2023-11-23 10:34:56|       72.35|      36.175|
| 2023-11-02 06:40:10|        22.4|        11.2|
| 2023-11-23 13:40:43|       36.12|       18.06|
| 2023-11-01 00:00:27|        61.0|        30.5|
| 2023-11-01 00:00:04|       21.36|       10.68|
| 2023-11-01 00:04:51|        34.8|        17.4|
| 2023-11-01 00:14:15|       21.36|       10.68|
+--------------------+------------+------------+



In [ ]:
# Option 1: Drop the WAP snapshot if validation fails
# Use expire_snapshots to remove the WAP snapshot without publishing it
# This discards the staged changes as if the operation never happened

# spark.sql(f"""
#     CALL polaris.system.expire_snapshots(
#         table => 'polaris.taxi.trips_by_month',
#         snapshot_ids => ARRAY({wap_snapshot_id})
#     )
# """)

In [20]:
# Option 2: Publish the WAP snapshot to make it visible to all readers
# This promotes the changes from staging to production

spark.sql("""
    CALL polaris.system.publish_changes(
        table => 'polaris.taxi.trips_by_month',
        wap_id => 'wap-demo-001'
    )
""")

spark.sql("""
    SELECT tpep_pickup_datetime, total_amount, correct_fare
    FROM polaris.taxi.trips_by_month
    LIMIT 10
""").show()

+--------------------+------------+------------+
|tpep_pickup_datetime|total_amount|correct_fare|
+--------------------+------------+------------+
| 2023-11-01 19:37:07|       46.85|      23.425|
| 2023-11-02 01:52:29|        13.6|         6.8|
| 2023-11-02 04:06:10|       101.5|       50.75|
| 2023-11-23 10:34:56|       72.35|      36.175|
| 2023-11-02 06:40:10|        22.4|        11.2|
| 2023-11-23 13:40:43|       36.12|       18.06|
| 2023-11-01 00:00:27|        61.0|        30.5|
| 2023-11-01 00:00:04|       21.36|       10.68|
| 2023-11-01 00:04:51|        34.8|        17.4|
| 2023-11-01 00:14:15|       21.36|       10.68|
+--------------------+------------+------------+



In [21]:
# Drop the correct_fare column before branching
# This ensures the branch has the same schema as the source parquet files

spark.sql("""
    ALTER TABLE polaris.taxi.trips_by_month
    DROP COLUMN correct_fare
""")

DataFrame[]

In [22]:
# Create a branch for experimentation
# This creates an isolated fork of the table history

spark.sql("""
    ALTER TABLE polaris.taxi.trips_by_month
    CREATE BRANCH experiment_branch
""")

DataFrame[]

In [23]:
# Insert data to the branch only
# Main branch remains unaffected

# Insert a few days of data from October into the branch
spark.sql("""
    INSERT INTO polaris.taxi.trips_by_month.branch_experiment_branch
    SELECT * FROM parquet.`s3a://warehouse/raw/yellow_tripdata_2023-10.parquet`
    WHERE tpep_pickup_datetime >= '2023-10-01'
      AND tpep_pickup_datetime < '2023-10-04'
""")

DataFrame[]

In [24]:
# Query the branch to verify changes - shows the 3 new rows
print("Branch (experiment_branch) with new data:")
spark.sql("""
    SELECT COUNT(*) as row_count
    FROM polaris.taxi.trips_by_month.branch_experiment_branch
""").show()

# Query the main branch - should not have the new rows
print("\nMain branch (unchanged):")
spark.sql("""
    SELECT COUNT(*) as row_count
    FROM polaris.taxi.trips_by_month
""").show()

Branch (experiment_branch) with new data:
+---------+
|row_count|
+---------+
| 15723193|
+---------+


Main branch (unchanged):
+---------+
|row_count|
+---------+
| 15407558|
+---------+



In [25]:
# Merge the branch back to main
# All commits from the branch become part of main's history

spark.sql("""
    CALL polaris.system.fast_forward(
        table => 'polaris.taxi.trips_by_month',
        branch => 'main',
        to => 'experiment_branch'
    )
""").show()

+--------------+-------------------+-------------------+
|branch_updated|       previous_ref|        updated_ref|
+--------------+-------------------+-------------------+
|          main|3487655135651362189|3547942540013582666|
+--------------+-------------------+-------------------+



In [26]:
# Query the branch to verify changes - shows the 3 new rows
print("Branch (experiment_branch) :")
spark.sql("""
    SELECT COUNT(*) as row_count
    FROM polaris.taxi.trips_by_month.branch_experiment_branch
""").show()

# Query the main branch 
print("\nMain branch:")
spark.sql("""
    SELECT COUNT(*) as row_count
    FROM polaris.taxi.trips_by_month
""").show()

Branch (experiment_branch) :
+---------+
|row_count|
+---------+
| 15723193|
+---------+


Main branch:
+---------+
|row_count|
+---------+
| 15723193|
+---------+



In [27]:
# Create a tag to mark an important snapshot
# Tags give snapshots memorable names instead of using snapshot IDs

spark.sql("""
    ALTER TABLE polaris.taxi.trips_by_month
    CREATE TAG end_of_quarter
""")

DataFrame[]

In [28]:
# Query using the tag
spark.sql("""
    SELECT COUNT(*) as trip_count
    FROM polaris.taxi.trips_by_month.tag_end_of_quarter
""").show()

+----------+
|trip_count|
+----------+
|  15723193|
+----------+



In [29]:
# Rollback to a previous snapshot for disaster recovery
# First, let's see the snapshot history

spark.sql("""
    SELECT 
        committed_at,
        snapshot_id,
        operation
    FROM polaris.taxi.trips_by_month.snapshots
    ORDER BY committed_at DESC
    LIMIT 5
""").show(truncate=False)

+-----------------------+-------------------+---------+
|committed_at           |snapshot_id        |operation|
+-----------------------+-------------------+---------+
|2026-10-20 05:39:26.927|3547942540013582666|append   |
|2026-10-20 05:27:59.908|3487655135651362189|overwrite|
|2026-10-20 05:21:56.505|1076862213835076847|append   |
+-----------------------+-------------------+---------+



In [30]:
# Rollback to the snapshot before the branch merge
# Get the second-most recent snapshot (before the merge)

from pyspark.sql.functions import col

# Get the snapshot before the current one
snapshots = spark.sql("""
    SELECT snapshot_id, committed_at, operation
    FROM polaris.taxi.trips_by_month.snapshots
    ORDER BY committed_at DESC
    LIMIT 2
""").collect()

previous_snapshot_id = snapshots[1]['snapshot_id']
print(f"Rolling back to snapshot: {previous_snapshot_id}")
print(f"Operation: {snapshots[1]['operation']}")

# Show Current Row Count
print("\nCurrent Row Count:")
spark.sql("""
    SELECT COUNT(*) as row_count
    FROM polaris.taxi.trips_by_month
""").show()

# Perform the rollback
spark.sql(f"""
    CALL polaris.system.rollback_to_snapshot(
        table => 'polaris.taxi.trips_by_month',
        snapshot_id => {previous_snapshot_id}
    )
""").show()

# Verify we're back to the original row count (before the merge)
print("\nRow count after rollback (should match original count before merge):")
spark.sql("""
    SELECT COUNT(*) as row_count
    FROM polaris.taxi.trips_by_month
""").show()

Rolling back to snapshot: 3487655135651362189
Operation: overwrite

Current Row Count:
+---------+
|row_count|
+---------+
| 15723193|
+---------+

+--------------------+-------------------+
|previous_snapshot_id|current_snapshot_id|
+--------------------+-------------------+
| 3547942540013582666|3487655135651362189|
+--------------------+-------------------+


Row count after rollback (should match original count before merge):
+---------+
|row_count|
+---------+
| 15407558|
+---------+



## Video: Schema Evolution for Iceberg Tables


In [31]:
# Setup: Create a fresh trips_by_day table for schema evolution demonstrations
spark.sql("DROP TABLE IF EXISTS polaris.taxi.trips_by_day")

spark.sql("""
    CREATE TABLE polaris.taxi.trips_by_day
    USING iceberg
    PARTITIONED BY (days(tpep_pickup_datetime))
    TBLPROPERTIES (
        'format-version' = '3',
        'write.target-file-size-bytes' = '16777216'
    )
    AS
    SELECT * FROM parquet.`s3a://warehouse/raw/`
""")

DataFrame[]

In [32]:
# Add a column to the table
spark.sql("""
    ALTER TABLE polaris.taxi.trips_by_day
    ADD COLUMN tip_percentage DOUBLE
""")

DataFrame[]

In [33]:
# Rename a column
spark.sql("""
    ALTER TABLE polaris.taxi.trips_by_day
    RENAME COLUMN tip_percentage TO tip_pct
""")

DataFrame[]

In [34]:
# Drop a column
spark.sql("""
    ALTER TABLE polaris.taxi.trips_by_day
    DROP COLUMN tip_pct
""")

DataFrame[]

## Video: Partition Evolution for Iceberg Tables

In [35]:
# Example: Change partition spec on an existing table
# Start with an unpartitioned table, then add day partitioning
spark.sql(""" DROP TABLE IF EXISTS polaris.taxi.trips_evolving """)
# First, create an unpartitioned table if it doesn't exist
spark.sql("""
    CREATE TABLE IF NOT EXISTS polaris.taxi.trips_evolving
    USING iceberg
    TBLPROPERTIES (
        'format-version' = '3'
    )
    AS SELECT * FROM parquet.`s3a://warehouse/raw/yellow_tripdata_2023-06.parquet`
""")

DataFrame[]

In [36]:
# View current partition spec
spark.sql("""
    DESCRIBE TABLE EXTENDED polaris.taxi.trips_evolving
""").show(100)

+--------------------+--------------------+-------+
|            col_name|           data_type|comment|
+--------------------+--------------------+-------+
|            VendorID|                 int|   NULL|
|tpep_pickup_datetime|       timestamp_ntz|   NULL|
|tpep_dropoff_date...|       timestamp_ntz|   NULL|
|     passenger_count|              bigint|   NULL|
|       trip_distance|              double|   NULL|
|          RatecodeID|              bigint|   NULL|
|  store_and_fwd_flag|              string|   NULL|
|        PULocationID|                 int|   NULL|
|        DOLocationID|                 int|   NULL|
|        payment_type|              bigint|   NULL|
|         fare_amount|              double|   NULL|
|               extra|              double|   NULL|
|             mta_tax|              double|   NULL|
|          tip_amount|              double|   NULL|
|        tolls_amount|              double|   NULL|
|improvement_surch...|              double|   NULL|
|        tot

In [37]:
# Add day partitioning to the table
# Old files remain unchanged; new writes will use day partitioning

spark.sql("""
    ALTER TABLE polaris.taxi.trips_evolving
    ADD PARTITION FIELD days(tpep_pickup_datetime)
""")

DataFrame[]

In [38]:
# View current partition spec
spark.sql("""
    DESCRIBE TABLE EXTENDED polaris.taxi.trips_evolving
""").show(100)

+--------------------+--------------------+-------+
|            col_name|           data_type|comment|
+--------------------+--------------------+-------+
|            VendorID|                 int|   NULL|
|tpep_pickup_datetime|       timestamp_ntz|   NULL|
|tpep_dropoff_date...|       timestamp_ntz|   NULL|
|     passenger_count|              bigint|   NULL|
|       trip_distance|              double|   NULL|
|          RatecodeID|              bigint|   NULL|
|  store_and_fwd_flag|              string|   NULL|
|        PULocationID|                 int|   NULL|
|        DOLocationID|                 int|   NULL|
|        payment_type|              bigint|   NULL|
|         fare_amount|              double|   NULL|
|               extra|              double|   NULL|
|             mta_tax|              double|   NULL|
|          tip_amount|              double|   NULL|
|        tolls_amount|              double|   NULL|
|improvement_surch...|              double|   NULL|
|        tot

In [39]:
# Insert new data - it will use the new partition spec
spark.sql("""
    INSERT INTO polaris.taxi.trips_evolving
    SELECT * FROM parquet.`s3a://warehouse/raw/yellow_tripdata_2023-07.parquet`
""")

DataFrame[]

In [40]:
# View partitions now - mix of unpartitioned and day-partitioned data
spark.sql("""
    SELECT partition FROM polaris.taxi.trips_evolving.partitions
""").show()

+------------+
|   partition|
+------------+
|{2023-07-22}|
|{2023-07-23}|
|{2023-09-25}|
|{2023-07-20}|
|{2023-07-21}|
|{2023-07-26}|
|{2023-07-27}|
|{2023-07-24}|
|{2023-07-25}|
|{2023-07-30}|
|{2023-07-31}|
|{2023-07-28}|
|{2023-07-29}|
|{2023-08-01}|
|      {NULL}|
|{2023-08-13}|
|{2023-10-16}|
|{2002-12-31}|
|{2023-07-02}|
|{2023-07-03}|
+------------+
only showing top 20 rows


In [41]:
# Partition reduction: Remove a partition transform
# This stops future writes from using that partition, but old files remain

spark.sql("""
    ALTER TABLE polaris.taxi.trips_evolving
    DROP PARTITION FIELD days(tpep_pickup_datetime)
""")

DataFrame[]

In [42]:
# Partition addition: Add multiple partition transforms
spark.sql("""
    ALTER TABLE polaris.taxi.trips_evolving
    ADD PARTITION FIELD months(tpep_pickup_datetime)
""")

DataFrame[]

In [43]:
# Insert new data - it will use the new partition spec
spark.sql("""
    INSERT INTO polaris.taxi.trips_evolving
    SELECT * FROM parquet.`s3a://warehouse/raw/yellow_tripdata_2023-08.parquet`
""")

DataFrame[]

In [44]:
spark.sql("REFRESH TABLE polaris.taxi.trips_evolving")

DataFrame[]

In [45]:
# View partitions now - mix of unpartitioned and day-partitioned data and month partitioned
spark.sql("""
    SELECT partition FROM polaris.taxi.trips_evolving.partitions
""").show()

+------------------+
|         partition|
+------------------+
|{2023-07-13, NULL}|
|{2023-07-27, NULL}|
|{2002-12-31, NULL}|
|{2023-07-10, NULL}|
|{2009-01-01, NULL}|
|{2023-07-24, NULL}|
|{2023-07-21, NULL}|
|{2023-07-18, NULL}|
|{2023-07-07, NULL}|
|{2023-08-01, NULL}|
|{2008-12-31, NULL}|
|{2023-07-22, NULL}|
|{2023-07-11, NULL}|
|{2023-07-08, NULL}|
|{2023-07-05, NULL}|
|       {NULL, 643}|
|{2023-07-30, NULL}|
|       {NULL, 644}|
|{2023-07-19, NULL}|
|{2023-08-13, NULL}|
+------------------+
only showing top 20 rows


## Video: Summary of Module 2

In [46]:
# Demonstrate automatic predicate transformation
# Query with timestamp filter - Iceberg automatically converts to partition filter

spark.sql("""
    SELECT 
        COUNT(*) as trip_count,
        MIN(tpep_pickup_datetime) as first_trip,
        MAX(tpep_pickup_datetime) as last_trip
    FROM polaris.taxi.trips_evolving
    WHERE tpep_pickup_datetime >= '2023-08-01 00:00:00'
      AND tpep_pickup_datetime < '2023-08-02 00:00:00'
""").show()

+----------+-------------------+-------------------+
|trip_count|         first_trip|          last_trip|
+----------+-------------------+-------------------+
|     98390|2023-08-01 00:00:00|2023-08-01 23:59:57|
+----------+-------------------+-------------------+

